In [27]:
import arcpy
from arcpy import env
import os
import numpy as np
from arcgis import GIS
from arcgis.features import GeoAccessor
from arcgis.features import GeoSeriesAccessor
import pandas as pd

arcpy.env.overwriteOutput = True
arcpy.env.parallelProcessingFactor = "90%"

# show all columns
pd.options.display.max_columns = None

# pd.pivot_table(df, values='a', index='b', columns='c', aggfunc='sum', fill_value=0)
# pd.DataFrame.spatial.from_featureclass(???)  
# df.spatial.to_featureclass(location=???,sanitize_columns=False)  

# gsa = arcgis.features.GeoSeriesAccessor(df['SHAPE'])  
# df['AREA'] = gsa.area  # KNOW YOUR UNITS

In [28]:
## spatial join
# target_features = ?
# join_features = ?
# output_features = os.path.join(gdb, ?)

# fieldmappings = arcpy.FieldMappings()
# fieldmappings.addTable(target_features)
# fieldmappings.addTable(join_features)

# # variable
# fieldindex = fieldmappings.findFieldMapIndex(?)
# fieldmap = fieldmappings.getFieldMap(fieldindex)
# fieldmap.mergeRule = 'Sum'
# fieldmappings.replaceFieldMap(fieldindex, fieldmap)

# sj = arcpy.SpatialJoin_analysis(target_features, join_features, output_features,'JOIN_ONE_TO_ONE', "KEEP_ALL", fieldmappings, match_option="INTERSECT")
# sj_df = pd.DataFrame.spatial.from_featureclass(sj[0]).copy()

In [29]:
# fill NA values in Spatially enabled dataframes (ignores SHAPE column)
def fill_na_sedf(df_with_shape_column, fill_value=0):
    if 'SHAPE' in list(df_with_shape_column.columns):
        cols_to_fill = df_with_shape_column.columns.difference(['SHAPE'])
        df_with_shape_column[cols_to_fill] = df_with_shape_column[cols_to_fill].fillna(fill_value)
        return df_with_shape_column
    else:
        raise Exception("Dataframe does not include 'SHAPE' column")

In [30]:
outputs = ['.\\Outputs', "scratch.gdb", 'results.gdb']

if not os.path.exists(outputs[0]):
    os.makedirs(outputs[0])

gdb = os.path.join(outputs[0], outputs[1])
gdb2 = os.path.join(outputs[0], outputs[2])

if not arcpy.Exists(gdb):
    arcpy.CreateFileGDB_management(outputs[0], outputs[1])

if not arcpy.Exists(gdb2):
    arcpy.CreateFileGDB_management(outputs[0], outputs[2])

## Load and Process Inputs

In [31]:
# year 
year = 2055

# county controls
county_controls = pd.read_csv("E:\Tasks\REMM-Manage-Base-Year-Data-2023\Inputs\County Controls\ControlTotal_SE_AllCounties.csv")

# se taz output
se_folder = r'E:\Tasks\REMM-Manage-Base-Year-Data-2023\Scripts\Outputs\SE_v10_20260204'

In [32]:
county_controls_year = county_controls[(county_controls['YEAR'] == year) & (county_controls['CO_NAME'].isin(['BOX ELDER - WFRC','WEBER - WFRC', 'DAVIS','SALT LAKE', 'UTAH']))].copy()

county_controls_year.rename({'HH':'TOTHH', 'CONS':'FM_CONS', 'MING':'FM_MING', 'AGRI':'FM_AGRI'}, axis=1, inplace=True)
county_controls_year['CO_NAME'] = county_controls_year['CO_NAME'].replace({'BOX ELDER - WFRC': 'BOX ELDER', 'WEBER - WFRC': 'WEBER'})

county_controls_year = county_controls_year.groupby('CO_NAME')[['TOTHH','HHPOP','ALLEMP','RETL','FOOD','MANU',
                                       'WSLE','OFFI','GVED','HLTH','OTHR',
                                       'FM_AGRI','FM_MING','FM_CONS','HBJ']].sum()

county_controls_year

,TOTHH,HHPOP,ALLEMP,RETL,FOOD,MANU,WSLE,OFFI,GVED,HLTH,OTHR,FM_AGRI,FM_MING,FM_CONS,HBJ
CO_NAME,,,,,,,,,,,,,,,
BOX ELDER,19089,43862,23823,2540,1265,3616,1246,1183,2870,1816,4329,962,77,2759,1160
DAVIS,209735,514648,280007,26678,11968,19141,12766,24881,48722,28126,74071,822,428,22032,10372
SALT LAKE,656012,1498704,1523546,117413,65850,85451,111791,225819,208081,134408,404406,1224,5204,102510,61389
UTAH,498253,1332420,687936,70816,32306,39484,26287,90904,89830,76112,167582,3637,1146,57237,32595
WEBER,146462,339121,208294,18990,9797,26386,10828,12486,31891,23889,49078,1343,184,16190,7232


In [33]:
se_output = pd.read_csv(os.path.join(se_folder, f'SE_{year}.csv'))
se_county_sum = se_output.groupby('CO_NAME')[['TOTHH','HHPOP','ALLEMP','RETL','FOOD','MANU',
                                       'WSLE','OFFI','GVED','HLTH','OTHR',
                                       'FM_AGRI','FM_MING','FM_CONS','HBJ']].sum()
se_county_sum

,TOTHH,HHPOP,ALLEMP,RETL,FOOD,MANU,WSLE,OFFI,GVED,HLTH,OTHR,FM_AGRI,FM_MING,FM_CONS,HBJ
CO_NAME,,,,,,,,,,,,,,,
BOX ELDER,19089.000000,4.386200e+04,2.322280e+04,2540.000000,1265.000000,3616.000000,1246.000000,1183.000000,2869.000000,1816.000000,4329.000000,962.000000,77.000000,2759.000000,1160.000000
DAVIS,209733.999969,5.146480e+05,2.800120e+05,26678.000010,11968.000001,19141.000002,12765.999999,24881.000025,48727.000366,28126.000051,74071.000032,822.000000,428.000000,22032.000015,10371.999998
SALT LAKE,595660.333040,1.498704e+06,1.523546e+06,117412.800032,65850.300119,85451.000119,111791.000098,225819.000047,208080.900096,134407.999989,404406.100287,1224.000000,5203.999996,102510.000017,61388.999969
UTAH,498099.767414,1.332133e+06,6.879314e+05,70816.200044,32305.700007,39483.999984,26286.999996,90903.799984,89825.500038,76111.700040,167582.500053,3636.999998,1146.000000,57237.000019,32595.000049
WEBER,135192.033312,3.390466e+05,2.082940e+05,18989.999997,9797.000000,26386.000001,10828.000001,12486.000001,31891.000001,23889.000002,49078.000004,1343.000000,184.000000,16190.000000,7231.999999


## Raw Difference

In [34]:
# + --> Too many
# - --> Too few
print(f'Year: {year}')
round(se_county_sum - county_controls_year)

Year: 2055


,TOTHH,HHPOP,ALLEMP,RETL,FOOD,MANU,WSLE,OFFI,GVED,HLTH,OTHR,FM_AGRI,FM_MING,FM_CONS,HBJ
CO_NAME,,,,,,,,,,,,,,,
BOX ELDER,-0.0,-0.0,-600.0,-0.0,0.0,0.0,-0.0,0.0,-1.0,-0.0,-0.0,-0.0,0.0,-0.0,-0.0
DAVIS,-1.0,-0.0,5.0,0.0,0.0,0.0,-0.0,0.0,5.0,0.0,0.0,0.0,0.0,0.0,-0.0
SALT LAKE,-60352.0,-0.0,0.0,-0.0,0.0,0.0,0.0,0.0,-0.0,-0.0,0.0,-0.0,-0.0,0.0,-0.0
UTAH,-153.0,-287.0,-5.0,0.0,-0.0,-0.0,-0.0,-0.0,-4.0,-0.0,1.0,-0.0,-0.0,0.0,0.0
WEBER,-11270.0,-74.0,0.0,-0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.0,0.0,-0.0,-0.0


## Percent Difference

In [35]:
# + --> Too many
# - --> Too few
print(f'Year: {year}')
round((se_county_sum - county_controls_year) / county_controls_year, 4)

Year: 2055


,TOTHH,HHPOP,ALLEMP,RETL,FOOD,MANU,WSLE,OFFI,GVED,HLTH,OTHR,FM_AGRI,FM_MING,FM_CONS,HBJ
CO_NAME,,,,,,,,,,,,,,,
BOX ELDER,-0.0000,-0.0000,-0.0252,-0.0,0.0,0.0,-0.0,0.0,-0.0003,-0.0,-0.0,-0.0,0.0,-0.0,-0.0
DAVIS,-0.0000,-0.0000,0.0000,0.0,0.0,0.0,-0.0,0.0,0.0001,0.0,0.0,0.0,0.0,0.0,-0.0
SALT LAKE,-0.0920,-0.0000,0.0000,-0.0,0.0,0.0,0.0,0.0,-0.0000,-0.0,0.0,-0.0,-0.0,0.0,-0.0
UTAH,-0.0003,-0.0002,-0.0000,0.0,-0.0,-0.0,-0.0,-0.0,-0.0001,-0.0,0.0,-0.0,-0.0,0.0,0.0
WEBER,-0.0769,-0.0002,0.0000,-0.0,0.0,0.0,0.0,0.0,0.0000,0.0,0.0,-0.0,0.0,-0.0,-0.0
